# 01 — Extract Shot Events
Load Wyscout event data league-by-league, filter shots, extract coordinates
and goal outcomes, then save combined, train, and test parquet files.

In [15]:
import json
import pandas as pd
from pathlib import Path
from IPython.display import display

In [16]:
import sys
!{sys.executable} -m pip install pyarrow



[notice] A new release of pip available: 22.3 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 0: Path validation

In [17]:
DATA_DIR = Path("../data/wyscout-data/events")
assert DATA_DIR.exists(), f"Missing directory: {DATA_DIR}"
assert (DATA_DIR / "events_England.json").exists(), "England events file not found"
print("Paths OK")

Paths OK


## Step 1: Helper functions

In [18]:
def extract_x(pos):
    return pos[0]["x"] if pos else None

def extract_y(pos):
    return pos[0]["y"] if pos else None

def extract_goal(tags):
    return int(any(tag["id"] == 101 for tag in tags))

## Step 2: Load shot events per league (memory-efficient)
Filter to shots at read time, then extract and cast coordinates and goal outcome immediately.

In [19]:
leagues = ["England", "France", "Germany", "Italy", "Spain"]
KEEP_COLS = [
    "league", "matchId", "playerId", "teamId",
    "eventSec", "matchPeriod", "subEventName",
    "positions", "tags", "x", "y", "is_goal"
]

events_by_league = {}

for league in leagues:
    with open(DATA_DIR / f"events_{league}.json", encoding="utf-8") as f:
        events = json.load(f)

    shots_only = [e for e in events if e["eventName"] == "Shot"]
    for e in shots_only:
        e["league"] = league

    league_df = pd.DataFrame(shots_only)
    league_df["x"]       = pd.to_numeric(league_df["positions"].apply(extract_x), errors="coerce")
    league_df["y"]       = pd.to_numeric(league_df["positions"].apply(extract_y), errors="coerce")
    league_df["is_goal"] = league_df["tags"].apply(extract_goal).astype("int8")

    events_by_league[league] = league_df[KEEP_COLS]
    print(f"{league}: {len(league_df):,} shots")

England: 8,451 shots
France: 8,327 shots
Germany: 6,898 shots
Italy: 8,806 shots
Spain: 7,979 shots


## Step 3: Build train/test DataFrames
Training leagues: France, Germany, Italy, Spain  
Test league: England (held out)

In [20]:
train_df = pd.concat(
    [events_by_league[l] for l in ["France", "Germany", "Italy", "Spain"]],
    ignore_index=True
)
test_df = events_by_league["England"].copy()

print(f"Train shots: {len(train_df):,}")
print(f"Test shots:  {len(test_df):,}")

Train shots: 32,010
Test shots:  8,451


## Step 4: Combine and preview

In [21]:
shots = pd.concat([train_df, test_df], ignore_index=True)
shots[["league", "subEventName", "x", "y", "is_goal"]].head()

,league,subEventName,x,y,is_goal
0,France,Shot,94,57,1
1,France,Shot,83,42,0
2,France,Shot,96,43,1
3,France,Shot,84,21,0
4,France,Shot,73,51,0


## Step 5: Sanity checks

> **Note:** This notebook extracts shot-level event data only.  
> Match dates and chronological ordering will be added in the next notebook using the matches files.

In [22]:
print("Missing coordinates:")
print(shots[["x", "y"]].isna().sum())

Missing coordinates:
x    0
y    0
dtype: int64


In [23]:
print("Coordinate ranges:")
print(f"x: {shots['x'].min()} to {shots['x'].max()}")
print(f"y: {shots['y'].min()} to {shots['y'].max()}")

Coordinate ranges:
x: 1 to 100
y: 0 to 100


In [24]:
print(shots[["x", "y", "is_goal"]].dtypes)

x          int64
y          int64
is_goal     int8
dtype: object


In [25]:
# Drop list-valued columns before deduplication — pandas cannot hash Python lists.
# Scalar identity fields (matchId, playerId, eventSec, etc.) are sufficient for this check.
print("Duplicate rows:", shots.drop(columns=["positions", "tags"]).duplicated().sum())

Duplicate rows: 0


In [26]:
print(f"Total shots: {len(shots):,}")
print(f"Goals:       {shots['is_goal'].sum():,}")
print(f"Goal rate:   {shots['is_goal'].mean():.2%}")

league_summary = shots.groupby("league")["is_goal"].agg(
    ["count", "sum", "mean"]
).rename(columns={"count": "shots", "sum": "goals", "mean": "goal_rate"})
print("\nLeague-level summary:")
display(league_summary)

Total shots: 40,461
Goals:       4,271
Goal rate:   10.56%

League-level summary:


,shots,goals,goal_rate
league,,,
England,8451,914,0.108153
France,8327,873,0.104840
Germany,6898,747,0.108292
Italy,8806,853,0.096866
Spain,7979,884,0.110791


## Step 6: Save outputs

In [27]:
OUT = Path("../data")
OUT.mkdir(parents=True, exist_ok=True)

shots.to_parquet(OUT / "wyscout_shots.parquet", index=False)
train_df.to_parquet(OUT / "wyscout_train_shots.parquet", index=False)
test_df.to_parquet(OUT / "wyscout_test_shots.parquet", index=False)
shots.head(100).to_csv(OUT / "wyscout_shots_sample.csv", index=False)

print(f"Saved {len(shots):,} combined shots  → wyscout_shots.parquet")
print(f"Saved {len(train_df):,} train shots   → wyscout_train_shots.parquet")
print(f"Saved {len(test_df):,} test shots    → wyscout_test_shots.parquet")
print("Saved 100-row sample         → wyscout_shots_sample.csv")

Saved 40,461 combined shots  → wyscout_shots.parquet
Saved 32,010 train shots   → wyscout_train_shots.parquet
Saved 8,451 test shots    → wyscout_test_shots.parquet
Saved 100-row sample         → wyscout_shots_sample.csv


In [28]:
check_df = pd.read_parquet(OUT / "wyscout_shots.parquet")
print(check_df.shape)
check_df.head()

(40461, 12)


,league,matchId,playerId,teamId,eventSec,matchPeriod,subEventName,positions,tags,x,y,is_goal
0,France,2500686,256992,3799,605.975493,1H,Shot,"[{'x': 94, 'y': 57}, {'x': 0, 'y': 0}]","[{'id': 101}, {'id': 402}, {'id': 201}, {'id':...",94,57,1
1,France,2500686,334552,3772,859.236394,1H,Shot,"[{'x': 83, 'y': 42}, {'x': 100, 'y': 100}]","[{'id': 403}, {'id': 1214}, {'id': 1802}]",83,42,0
2,France,2500686,26389,3772,1568.104834,1H,Shot,"[{'x': 96, 'y': 43}, {'x': 100, 'y': 100}]","[{'id': 101}, {'id': 402}, {'id': 201}, {'id':...",96,43,1
3,France,2500686,276920,3772,1800.852078,1H,Shot,"[{'x': 84, 'y': 21}, {'x': 100, 'y': 100}]","[{'id': 402}, {'id': 1210}, {'id': 1802}]",84,21,0
4,France,2500686,366760,3799,2009.537139,1H,Shot,"[{'x': 73, 'y': 51}, {'x': 0, 'y': 0}]","[{'id': 402}, {'id': 201}, {'id': 1203}, {'id'...",73,51,0
